In [ ]:
# ============================================
# Startup Cell: Mount Drive + Import Config
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import sys
import os

# -------------------------------------------------
# Project base directory (Drive)
# -------------------------------------------------
BASE_DRIVE_DIR = "/content/drive/MyDrive/DIP_Project"

# -------------------------------------------------
# Add src/ to Python path and import config
# -------------------------------------------------
sys.path.append(f"{BASE_DRIVE_DIR}/src")

from project_config import *

# -------------------------------------------------
# Basic environment confirmation
# -------------------------------------------------
print("Drive mounted successfully.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"METADATA_DIR: {METADATA_DIR}")
print(f"PROCESSED_DATA_DIR: {PROCESSED_DATA_DIR}")
print(f"LOCAL_WORK_DIR: {LOCAL_WORK_DIR}")



In [ ]:
# ============================================
# Cell 0: Notebook Overview
# ============================================
# Purpose:
#   This notebook verifies and applies frequency-domain feature extraction
#   for a selected dataset subset using metadata-driven image selection
#   and preprocessed grayscale images stored on Google Drive.
#
# Inputs:
#   The notebook expects:
#     - a subset-specific metadata CSV file stored in:
#         data/metadata/
#       such as:
#         train_metadata.csv
#         test_metadata.csv
#     - dataset-specific preprocessed image folders stored in:
#         data/preprocessed/<dataset>/images/
#
# Assumptions:
#   - All images have already been preprocessed.
#   - All images are already grayscale.
#   - All images have already been resized to 256 x 256.
#   - This notebook does NOT perform resizing or grayscale conversion.
#   - Input selection is metadata-driven.
#   - Class labels are taken from metadata, not inferred from filenames.
#   - The notebook uses project_config.py as the central source for
#     directory paths, file names, and shared project constants.
#   - This notebook focuses only on frequency-domain feature extraction.
#
# What the notebook does:
#   Startup Cell:
#     Mount Google Drive, import project_config.py, and initialize
#     the notebook environment.
#
#   Cell 1:
#     Import required libraries for image loading, numerical processing,
#     Fourier analysis, feature extraction, and visualization.
#
#   Cell 2:
#     Define the selected subset, input metadata path, preprocessed image
#     root directory, output CSV path, and dataset-folder mapping.
#
#   Cell 3:
#     Verify required inputs exist before processing:
#       - confirm the selected metadata CSV exists
#       - confirm the preprocessed root directory exists
#       - confirm metadata has expected columns
#       - validate subset values
#       - validate source_dataset values
#       - resolve and verify a sample image path
#
#   Cell 4:
#     Load the selected sample image and verify its properties,
#     including:
#       - shape
#       - datatype
#       - intensity range
#
#   Cell 5:
#     Define helper functions for:
#       - 2D Fourier transform computation
#       - centered magnitude and power spectrum computation
#       - radial profile computation
#       - frequency-band energy ratio computation
#       - frequency-domain feature extraction
#
#   Cell 6:
#     Compute intermediate frequency-domain representations for the
#     sample image, including:
#       - log-magnitude spectrum
#       - power spectrum
#       - radial energy profile
#       - frequency-domain feature values
#
#   Cell 7:
#     Display intermediate visual results for the sample image:
#       - input image
#       - centered log-magnitude spectrum
#       - power spectrum visualization
#       - radial profile plot
#
#   Cell 8:
#     Plot histograms related to:
#       - log-magnitude spectrum values
#       - radial energy values
#
#   Cell 9:
#     Compare a small set of real and AI-generated images from the
#     selected subset and print their frequency-domain feature values.
#
#   Cell 10:
#     Apply frequency-domain feature extraction to all images identified
#     by the selected metadata CSV and save the results to a subset-
#     specific output CSV.
#
#   Cell 11:
#     Reload and verify the saved frequency feature CSV.
#
# Outputs:
#   Primary outputs:
#     - visual validation of frequency-domain feature extraction behavior
#     - printed frequency-domain feature values for selected images
#     - subset-specific frequency feature CSV
#
# Notes:
#   - This notebook supports exploratory validation and full-subset feature
#     extraction for a selected subset (train or test).
#   - Only frequency-domain features are extracted in this notebook.
#   - Later notebooks merge gradient, spatial, and frequency-domain
#     features into complete feature vectors for classifier training
#     and evaluation.
#   - This notebook is intended to follow the same structural pattern
#     as the other feature-extraction notebooks for consistency across
#     the project pipeline.
#   - This notebook must be executed separately for each subset:
#       1. Set SUBSET_NAME = TRAIN_SUBSET and run all cells to generate:
#            train_frequency_features.csv
#       2. Set SUBSET_NAME = TEST_SUBSET and run all cells to generate:
#            test_frequency_features.csv
#
#   - The training and test feature files must remain strictly separate.
#     The training set is used for model development, while the test set
#     is reserved for final evaluation only.
# ============================================

print("Notebook overview loaded.")



In [ ]:
# ============================================
# Cell 1: Import Required Libraries
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm.notebook import tqdm

from PIL import Image
from scipy.stats import entropy

print("Libraries imported successfully.")



In [ ]:
# ============================================
# Cell 2: Define Subset and Paths
# ============================================

SUBSET_NAME = TRAIN_SUBSET

if SUBSET_NAME == TRAIN_SUBSET:
    INPUT_METADATA_CSV = os.path.join(METADATA_DIR, TRAIN_METADATA_FILENAME)
    OUTPUT_FEATURES_CSV = os.path.join(METADATA_DIR, "train_frequency_features.csv")
elif SUBSET_NAME == TEST_SUBSET:
    INPUT_METADATA_CSV = os.path.join(METADATA_DIR, TEST_METADATA_FILENAME)
    OUTPUT_FEATURES_CSV = os.path.join(METADATA_DIR, "test_frequency_features.csv")
else:
    raise ValueError(f"Unsupported subset: {SUBSET_NAME}")

# Base directory containing dataset-specific preprocessed image folders
PREPROCESSED_ROOT_DIR = PROCESSED_DATA_DIR

# Map metadata source_dataset values to actual folder names in Drive
SOURCE_DATASET_TO_FOLDER = {
    "DiffusionDB": "DiffusionDB",
    "SDXL_Generated_10K": "SDXL_Generated_10K",
    "Midjourney": "Midjourney",
    "ImageNet_1K_256": "ImageNet_1K_256",
    "MS_COCO_2017": "MS_COCO_2017",
    "OpenImages": "OpenImages",
}

SAMPLE_IMAGE_PATH = None
SAMPLE_IMAGE_NAME = None

print(f"Selected subset: {SUBSET_NAME}")
print(f"Input metadata CSV: {INPUT_METADATA_CSV}")
print(f"Preprocessed root directory: {PREPROCESSED_ROOT_DIR}")
print(f"Output feature CSV: {OUTPUT_FEATURES_CSV}")



In [ ]:
# ============================================
# Cell 3: Verify Inputs and Select Sample Image
# ============================================

required_metadata_columns = [
    "filename",
    "class_label",
    "source_dataset",
    "subset",
]

print("Validating inputs...")

# --------------------------------------------
# Check metadata CSV
# --------------------------------------------
if not os.path.exists(INPUT_METADATA_CSV):
    raise FileNotFoundError(f"Missing metadata CSV: {INPUT_METADATA_CSV}")

# --------------------------------------------
# Check preprocessed root directory
# --------------------------------------------
if not os.path.isdir(PREPROCESSED_ROOT_DIR):
    raise FileNotFoundError(
        f"Missing preprocessed root directory: {PREPROCESSED_ROOT_DIR}"
    )

# --------------------------------------------
# Load metadata
# --------------------------------------------
df_meta = pd.read_csv(INPUT_METADATA_CSV)

# --------------------------------------------
# Validate required columns
# --------------------------------------------
missing_columns = [
    col for col in required_metadata_columns
    if col not in df_meta.columns
]

if missing_columns:
    raise ValueError(f"Missing required metadata columns: {missing_columns}")

# --------------------------------------------
# Validate subset
# --------------------------------------------
if not all(df_meta["subset"] == SUBSET_NAME):
    raise ValueError(
        f"Metadata contains rows outside subset '{SUBSET_NAME}'"
    )

# --------------------------------------------
# Validate source_dataset values
# --------------------------------------------
unknown_sources = [
    s for s in df_meta["source_dataset"].unique()
    if s not in SOURCE_DATASET_TO_FOLDER
]

if unknown_sources:
    raise ValueError(f"Unknown source_dataset values: {unknown_sources}")

# --------------------------------------------
# Helper: build full image path
# --------------------------------------------
def get_image_path(row):
    return os.path.join(
        PREPROCESSED_ROOT_DIR,
        SOURCE_DATASET_TO_FOLDER[row["source_dataset"]],
        "images",
        row["filename"],
    )

# --------------------------------------------
# Select sample image
# --------------------------------------------
sample_row = df_meta.iloc[0]
SAMPLE_IMAGE_NAME = sample_row["filename"]
SAMPLE_IMAGE_PATH = get_image_path(sample_row)

if not os.path.exists(SAMPLE_IMAGE_PATH):
    raise FileNotFoundError(f"Sample image not found: {SAMPLE_IMAGE_PATH}")

# --------------------------------------------
# Minimal confirmation output
# --------------------------------------------
print(f"Rows: {len(df_meta)}")
display(df_meta.head())


In [ ]:
# ============================================
# Cell 4: Load Sample Image and Verify Properties
# ============================================

sample_image = Image.open(SAMPLE_IMAGE_PATH)
sample_array = np.array(sample_image)

print("Sample image loaded successfully.")
print(f"Sample image name: {SAMPLE_IMAGE_NAME}")
print(f"Sample image path: {SAMPLE_IMAGE_PATH}")
print(f"Shape: {sample_array.shape}")
print(f"Data type: {sample_array.dtype}")
print(f"Min intensity: {sample_array.min()}")
print(f"Max intensity: {sample_array.max()}")

plt.figure(figsize=(5, 5))
plt.imshow(sample_array, cmap="gray")
plt.title(f"Sample Image: {SAMPLE_IMAGE_NAME}")
plt.axis("off")
plt.show()



In [ ]:
# ============================================
# Cell 5: Frequency Feature Helper Functions
# ============================================

def compute_fft(img):
    F = np.fft.fft2(img)
    F_shift = np.fft.fftshift(F)
    return F_shift


def compute_magnitude_spectrum(F_shift):
    mag = np.abs(F_shift)
    log_mag = np.log1p(mag)
    power = mag ** 2
    return mag, log_mag, power


def radial_profile(power_spectrum):
    h, w = power_spectrum.shape
    cy, cx = h // 2, w // 2

    y, x = np.indices((h, w))
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    r = r.astype(np.int32)

    max_r = r.max()
    radial_mean = np.zeros(max_r + 1, dtype=np.float32)

    for i in range(max_r + 1):
        mask = (r == i)
        if np.any(mask):
            radial_mean[i] = power_spectrum[mask].mean()

    return radial_mean


def radial_entropy(radial_vals):
    hist, _ = np.histogram(radial_vals, bins=64)
    hist = hist.astype(np.float64)
    hist = hist / (hist.sum() + 1e-12)
    hist = np.clip(hist, 1e-12, None)
    return float(entropy(hist, base=2))


def frequency_band_ratios(power_spectrum):
    h, w = power_spectrum.shape
    cy, cx = h // 2, w // 2

    y, x = np.indices((h, w))
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)

    max_r = np.max(r)

    low_mask = r <= max_r * 0.2
    mid_mask = (r > max_r * 0.2) & (r <= max_r * 0.5)
    high_mask = r > max_r * 0.5

    total_energy = power_spectrum.sum() + 1e-12

    low_ratio = float(power_spectrum[low_mask].sum() / total_energy)
    mid_ratio = float(power_spectrum[mid_mask].sum() / total_energy)
    high_ratio = float(power_spectrum[high_mask].sum() / total_energy)

    return low_ratio, mid_ratio, high_ratio


def spectral_centroid(radial_vals):
    indices = np.arange(len(radial_vals))
    total = radial_vals.sum() + 1e-12
    centroid = float((indices * radial_vals).sum() / total)
    return centroid


def spectral_bandwidth(radial_vals, centroid):
    indices = np.arange(len(radial_vals))
    total = radial_vals.sum() + 1e-12
    variance = ((indices - centroid) ** 2 * radial_vals).sum() / total
    return float(np.sqrt(variance))


def extract_frequency_features(img):
    F_shift = compute_fft(img)
    mag, log_mag, power = compute_magnitude_spectrum(F_shift)

    radial_vals = radial_profile(power)

    low_ratio, mid_ratio, high_ratio = frequency_band_ratios(power)

    r_mean = float(np.mean(radial_vals))
    r_std = float(np.std(radial_vals))
    r_entropy = radial_entropy(radial_vals)

    centroid = spectral_centroid(radial_vals)
    bandwidth = spectral_bandwidth(radial_vals, centroid)

    log_std = float(np.std(log_mag))

    features = {
        "Low Frequency Energy Ratio": low_ratio,
        "High Frequency Energy Ratio": high_ratio,
        "Radial Mean": r_mean,
        "Radial Std": r_std,
        "Radial Entropy": r_entropy,
        "Spectral Centroid": centroid,
        "Spectral Bandwidth": bandwidth,
        "Log Spectrum Std": log_std,
    }

    return features, log_mag, power, radial_vals



In [ ]:
# ============================================
# Cell 6: Compute Frequency Components
# ============================================

features, log_mag, power, radial_vals = extract_frequency_features(sample_array)

print(f"Frequency components computed for sample image: {SAMPLE_IMAGE_NAME}")
print("Log magnitude shape  =", log_mag.shape)
print("Power spectrum shape =", power.shape)
print("Radial profile length =", len(radial_vals))



In [ ]:
# ============================================
# Cell 7: Display Frequency Visualization Results
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(sample_array, cmap="gray")
axes[0].set_title(f"Input Image: {SAMPLE_IMAGE_NAME}")
axes[0].axis("off")

im1 = axes[1].imshow(log_mag, cmap="gray")
axes[1].set_title("Centered Log-Magnitude Spectrum")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(np.log1p(power), cmap="gray")
axes[2].set_title("Log Power Spectrum")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle("Frequency-Domain Visualization", fontsize=12)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(radial_vals)
plt.title("Radial Energy Profile")
plt.xlabel("Radius")
plt.ylabel("Mean Power")
plt.tight_layout()
plt.show()



In [ ]:
# ============================================
# Cell 8: Plot Frequency Histograms / Profiles
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(log_mag.ravel(), bins=64)
axes[0].set_title("Log-Magnitude Spectrum Histogram")
axes[0].set_xlabel("Log-Magnitude")
axes[0].set_ylabel("Count")

axes[1].hist(radial_vals, bins=64)
axes[1].set_title("Radial Energy Histogram")
axes[1].set_xlabel("Radial Mean Power")
axes[1].set_ylabel("Count")

plt.suptitle(f"Frequency Histograms: {SAMPLE_IMAGE_NAME}", fontsize=12)
plt.tight_layout()
plt.show()



In [ ]:
# ============================================
# Cell 9: Compare Real vs AI Images
# ============================================

df_real = df_meta[df_meta["class_label"] == "rl"].sample(
    n=min(2, len(df_meta[df_meta["class_label"] == "rl"])),
    random_state=42
)

df_ai = df_meta[df_meta["class_label"] == "ai"].sample(
    n=min(2, len(df_meta[df_meta["class_label"] == "ai"])),
    random_state=42
)

sample_df = pd.concat([df_real, df_ai], axis=0).reset_index(drop=True)

print(f"Selected images from {SUBSET_NAME} subset:")
print(sample_df[["filename", "class_label", "source_dataset"]].to_string(index=False))
print("\nClass counts:")
print(sample_df["class_label"].value_counts().to_string())
print(f"\nTotal selected: {len(sample_df)}")

for i, row in sample_df.iterrows():
    fname = row["filename"]
    label = row["class_label"]
    source = row["source_dataset"]

    image_path = get_image_path(row)

    img_sample = np.array(Image.open(image_path))
    features_sample, log_mag_s, power_s, radial_vals_s = extract_frequency_features(img_sample)

    print("\n============================================")
    print(f"{SUBSET_NAME.upper()} | Image {i+1} of {len(sample_df)}")
    print(f"Filename: {fname}")
    print(f"Label: {label} | Source: {source}")

    for k, v in features_sample.items():
        print(f"{k:25s}: {v:.6f}")

    plt.figure(figsize=(8, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img_sample, cmap="gray")
    plt.title(f"Image ({label})")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(log_mag_s, cmap="gray")
    plt.title("Log-Magnitude Spectrum")
    plt.axis("off")

    plt.suptitle(fname, fontsize=10)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================
# Cell 10: Batch Frequency Feature Extraction
# ============================================

rows = []
skipped = 0

for _, row in tqdm(
    df_meta.iterrows(),
    total=len(df_meta),
    desc=f"{SUBSET_NAME} frequency features"
):
    fname = row["filename"]
    image_path = get_image_path(row)

    try:
        img_batch = np.array(Image.open(image_path))

        features_batch, _, _, _ = extract_frequency_features(img_batch)

        out_row = row.to_dict()
        out_row.update(features_batch)
        rows.append(out_row)

    except Exception as e:
        skipped += 1
        print(f"Skipping {fname}: {e}")

# Create DataFrame
features_df = pd.DataFrame(rows)

# Save to CSV
features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

# Summary
print(f"Saved: {OUTPUT_FEATURES_CSV}")
print(f"Expected rows: {len(df_meta)}")
print(f"Extracted rows: {len(features_df)}")
print(f"Skipped rows: {skipped}")
print(f"Processed subset: {SUBSET_NAME}")



In [ ]:
# ============================================
# Cell 11: Verify Saved Frequency Feature CSV
# ============================================

df_check = pd.read_csv(OUTPUT_FEATURES_CSV)

print("Saved frequency feature CSV verified.")
print(f"File: {OUTPUT_FEATURES_CSV}")
print(f"Shape: {df_check.shape}")

print("\nColumns:")
print(df_check.columns.tolist())

print("\nFirst 5 rows:")
display(df_check.head())

